In [2]:
import geopandas as gpd
import pandas as pd

In [ ]:
gdf_kd = gpd.read_file("Data_Processing/rnpd.geojson")    #crs: epsg:4326

In [ ]:
gdf_regije = gpd.read_file("Data_Processing/RazdelitevSlo/SR.geojson")

In [ ]:
gdf_poplave_zelo_redke = gpd.read_file("Data_Processing/DRSV_OPKP_ZR_POPL/DRSV_OPKP_ZR_POPL.shp")
gdf_poplave_redke = gpd.read_file("Data_Processing/DRSV_OPKP_REDKE_POPL/DRSV_OPKP_REDKE_POPL.shp")
gdf_poplave_pogoste = gpd.read_file("Data_Processing/DRSV_OPKP_POGOSTE_POPL/DRSV_OPKP_POGOSTE_POPL.shp")
gdf_pozarna_ogrozenost = gpd.read_file("Data_Processing/pozarna_ogrozenost_majhen_100m.geojson")
gdf_plazovi = gpd.read_file("Data_Processing/DRSV_Opk_Plazovi_skupna/plazovi2.shp")

gdf_zrak_postaje = gpd.read_file("Data_Processing/zrak_postaje.geojson")

In [6]:
gdfs = [gdf_kd, gdf_regije, gdf_poplave_zelo_redke, gdf_poplave_redke, gdf_poplave_pogoste, gdf_pozarna_ogrozenost, gdf_plazovi, gdf_zrak_postaje]

In [7]:
gdf_regije = gdf_regije.to_crs(epsg=4326)
gdf_poplave_zelo_redke = gdf_poplave_zelo_redke.to_crs(epsg=4326)
gdf_poplave_redke = gdf_poplave_redke.to_crs(epsg=4326)
gdf_poplave_pogoste = gdf_poplave_pogoste.to_crs(epsg=4326)
gdf_pozarna_ogrozenost = gdf_pozarna_ogrozenost.to_crs(epsg=4326)
gdf_plazovi = gdf_plazovi.to_crs(epsg=4326)
gdf_zrak_postaje = gdf_zrak_postaje.to_crs(epsg=4326)

In [8]:
gdf_poplave_pogoste = gdf_poplave_pogoste.rename(columns={"PP_ID": "ID", "PP_IME": "IME"})
gdf_poplave_redke = gdf_poplave_redke.rename(columns={"RP_ID": "ID", "RP_IME": "IME"})
gdf_poplave_zelo_redke = gdf_poplave_zelo_redke.rename(columns={"ZR_ID": "ID", "ZR_IME": "IME"})

#popravimo da so isti tipi vseh stolpcev
gdf_poplave_zelo_redke["OC_ZAN"] = pd.to_numeric(gdf_poplave_zelo_redke["OC_ZAN"], errors="coerce")
gdf_poplave_zelo_redke["DAT_KON"] = pd.to_datetime(gdf_poplave_zelo_redke["DAT_KON"], errors="coerce")
gdf_poplave_zelo_redke["DATP_KON"] = pd.to_datetime(gdf_poplave_zelo_redke["DATP_KON"], errors="coerce")

gdf_poplave_zelo_redke["OBJECTID_1"] = pd.NA

gdf_poplave_pogoste["flood_level"] = 3       
gdf_poplave_redke["flood_level"] = 2         
gdf_poplave_zelo_redke["flood_level"] = 1

skupaj_poplave = gpd.GeoDataFrame(
    pd.concat(
        [gdf_poplave_pogoste, gdf_poplave_redke, gdf_poplave_zelo_redke],
        ignore_index=True
    ),
    crs=gdf_poplave_pogoste.crs
)

In [9]:
def spatial_join_attr(base, polygons, col):
    joined = base.sjoin(polygons[['geometry', col]], how='left', predicate='within')
    return joined.groupby(level=0)[col].first()    #groupy daa je le en rezultat ce pade v vec polygonov

In [10]:
gdf_kd['poplave_ocena'] = None
gdf_kd['poplave_ocena'] = gdf_kd['poplave_ocena'].astype(float)

gdf_kd['poplave'] = pd.NA
gdf_kd['poplave'] = spatial_join_attr(gdf_kd, skupaj_poplave, 'flood_level')
gdf_kd['pozar'] = spatial_join_attr(gdf_kd, gdf_pozarna_ogrozenost, 'pozar')
gdf_kd['plazovi'] = spatial_join_attr(gdf_kd, gdf_plazovi, 'DN')

In [11]:
gdf_kd = gdf_kd.to_crs(epsg=3857)
skupaj_poplave = skupaj_poplave.to_crs(epsg=3857)

nearest = gpd.sjoin_nearest(
    gdf_kd[gdf_kd['poplave'].isna()][['geometry']],  # only geometry, nothing else
    skupaj_poplave[['flood_level', 'geometry']],
    max_distance=1000,
    distance_col='razdalja_pop',
    how='left'
)

gdf_kd.loc[nearest.index, 'poplave'] = nearest['flood_level'].values

In [42]:
gdf_kd = gdf_kd.to_crs(epsg=3857)
gdf_pozarna_ogrozenost = gdf_pozarna_ogrozenost.to_crs(epsg=3857)
gdf_kd['pozar'] = pd.to_numeric(gdf_kd['pozar'], errors='coerce')

nearest1 = gpd.sjoin_nearest(
    gdf_kd[gdf_kd['pozar'].isna()][['geometry']],
    gdf_pozarna_ogrozenost[['pozar', 'geometry']],
    max_distance=1000,
    how='left'
)

In [44]:
nearest1['pozar'] = pd.to_numeric(nearest1['pozar'], errors='coerce')
gdf_kd.loc[nearest1.index, 'pozar'] = nearest1['pozar'].values

In [37]:
gdf_kd = gdf_kd.to_crs(epsg=3857)
gdf_plazovi = gdf_plazovi.to_crs(epsg=3857)
gdf_plazovi['plazovi'] = gdf_plazovi['DN']

nearest2 = gpd.sjoin_nearest(
    gdf_kd[gdf_kd['plazovi'].isna()][['geometry']],
    gdf_plazovi[['plazovi', 'geometry']],
    max_distance=1000,
    how='left'
)

In [45]:
gdf_kd.loc[nearest2.index, 'plazovi'] = nearest2['plazovi'].values

In [46]:
gdf_kd = gdf_kd.to_crs(epsg=4326)

In [47]:
gdf_kd['regija'] = spatial_join_attr(gdf_kd, gdf_regije, 'SR_UIME')

In [48]:
gdf_kd.loc[gdf_kd['regija'].isna(), 'regija'] = "Obalno-kraška"   #vse, ki so bile nan so v tržaškem zalivu/na obali

In [49]:
gdf_kd['pozar'] = 5 - gdf_kd['pozar']

In [57]:
print(gdf_kd['plazovi'].value_counts())
print(gdf_kd['pozar'].value_counts())
print(gdf_kd['poplave'].value_counts())

plazovi
3.0    6502
0.0    6006
4.0    2701
2.0    1121
1.0     178
Name: count, dtype: int64
pozar
4.0    13529
3.0    11920
2.0     2252
1.0     1589
Name: count, dtype: int64
poplave
1.0    11468
2.0     3056
3.0      576
Name: count, dtype: int64


In [ ]:
gdf_kd.to_crs(epsg=4326).to_file('Data/brez_filanja_lukenj.geojson', driver='GeoJSON')